# Poster Figures (v3) — Feature Columns + Full Sharpe Overview

Fixes requested:
- Feature sets should show **specific feature columns**, not only feature groups
- Sharpe overview should show **ALL configs** (not just top-N) and clearly highlight **Logistic + 3D**

Prereq (run once):
```bash
python -m src.pipeline --mode live --notebook 02
```


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
ROOT = HERE if (HERE / 'outputs').exists() else HERE.parent
OUT_METRICS = ROOT / 'outputs' / 'metrics'
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('OUT_METRICS =', OUT_METRICS)
print('OUT_FIG =', OUT_FIG)


## 1) Feature Group Map (group → specific columns)
This matches the screenshot you shared.

In [ ]:
from src.features import feature_group_map

fg = feature_group_map()
feature_map_df = pd.DataFrame([
    {'Feature Group': k, 'Included Feature Columns': ', '.join(v)}
    for k, v in fg.items()
]).sort_values('Feature Group').reset_index(drop=True)
feature_map_df

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.axis('off')
tbl = ax.table(
    cellText=feature_map_df.values,
    colLabels=feature_map_df.columns,
    cellLoc='left',
    colLoc='left',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.35)
ax.set_title('Feature Group Map: Group → Specific Feature Columns', fontsize=14, pad=14)
plt.tight_layout()
out_path = OUT_FIG / 'poster_feature_group_map_full.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 2) Feature Set Map (F1–F5 → specific columns)
This is what you asked for: each feature set expanded to the **exact columns** included.

In [ ]:
from src.config import AppConfig

config = AppConfig()
order = ['F1_momentum','F2_momentum_reversal','F3_plus_risk_liquidity','F4_plus_cross_sectional','F5_full_finance']

def expand_groups_to_cols(groups: list[str]) -> list[str]:
    cols = []
    for g in groups:
        cols.extend(fg.get(g, []))
    # unique preserve order
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

rows = []
for fs in order:
    groups = config.feature_sets[fs]
    cols = expand_groups_to_cols(groups)
    rows.append({
        'Feature Set': fs,
        'Groups': ', '.join(groups),
        'Specific Feature Columns': ', '.join(cols),
        'n_cols': len(cols),
    })

feature_set_cols_df = pd.DataFrame(rows)
feature_set_cols_df

In [ ]:
# Export BIG PNG table (Canva)
# This is wide by nature; in Canva you can scale and crop if needed.
show_df = feature_set_cols_df[['Feature Set','Specific Feature Columns','n_cols']].copy()

fig, ax = plt.subplots(figsize=(18, 8))
ax.axis('off')
tbl = ax.table(
    cellText=show_df.values,
    colLabels=show_df.columns,
    cellLoc='left',
    colLoc='left',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.4)
ax.set_title('Feature Sets Expanded: F1–F5 → Specific Feature Columns', fontsize=14, pad=14)
plt.tight_layout()
out_path = OUT_FIG / 'poster_feature_sets_specific_columns.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 3) Load backtest CSV and show Sharpe across ALL configs

In [ ]:
backtest_path = OUT_METRICS / '02_live_backtest_summary.csv'
assert backtest_path.exists(), f'Missing: {backtest_path} (run pipeline first)'
backtest = pd.read_csv(backtest_path)

# sanity: what horizons are present?
print('Unique horizons:', sorted(backtest['horizon_days'].unique().tolist()))
print('Unique models:', sorted(backtest['model'].unique().tolist()))
backtest[['feature_set','model','horizon_days','sharpe']].head()

### 3A) Sharpe heatmap (Feature Set × (Model,Horizon))
This compactly shows **everything at once** and makes Logistic+3D easy to spot.

In [ ]:
bt = backtest.copy()
bt['col'] = bt.apply(lambda r: f"{r['model']}|{int(r['horizon_days'])}D", axis=1)
pivot = bt.pivot_table(index='feature_set', columns='col', values='sharpe', aggfunc='mean')
# order rows
pivot = pivot.reindex(order)
pivot

In [ ]:
import numpy as np

plt.figure(figsize=(12, 4.5))
plt.imshow(pivot.values, aspect='auto')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha='right')
plt.yticks(range(len(pivot.index)), pivot.index)
# annotate
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.iloc[i,j]
        txt = 'NA' if pd.isna(v) else f'{v:.2f}'
        plt.text(j, i, txt, ha='center', va='center', fontsize=9, color='black')
plt.title('Sharpe Heatmap: Feature Set × (Model | Horizon)')
plt.colorbar(label='Sharpe')
plt.tight_layout()
out_path = OUT_FIG / 'poster_sharpe_heatmap_all_configs.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

### 3B) Full ranked bar chart (ALL configs)
If you prefer bars: we plot **all configs** and highlight Logistic+3D in blue.

In [ ]:
all_bt = bt.copy()
all_bt['label'] = all_bt.apply(lambda r: f"{r['feature_set']} | {r['model']} | {int(r['horizon_days'])}D", axis=1)
all_bt = all_bt.sort_values('sharpe', ascending=True).reset_index(drop=True)

colors = []
for _, r in all_bt.iterrows():
    if (r['model'] == 'logistic_regression') and (int(r['horizon_days']) == 3):
        colors.append('#2563eb')
    else:
        colors.append('#9ca3af')

plt.figure(figsize=(12, 6))
plt.barh(all_bt['label'], all_bt['sharpe'], color=colors)
plt.xlabel('Sharpe')
plt.title('Sharpe Across ALL Configurations (blue = Logistic + 3D)')
plt.tight_layout()
out_path = OUT_FIG / 'poster_sharpe_all_configs_full.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## What to upload into Canva
- `outputs/figures/poster_feature_group_map_full.png`
- `outputs/figures/poster_feature_sets_specific_columns.png`  (this is the **feature set → specific columns** figure)
- `outputs/figures/02_live_rankic_heatmap.png` (existing)
- `outputs/figures/02_live_F5_full_finance_logistic_regression_3d_equity.png` (existing)
- `outputs/figures/poster_sharpe_heatmap_all_configs.png` (compact best option)
- `outputs/figures/poster_sharpe_all_configs_full.png` (bar chart of all configs, more crowded)
